**1. Installation & Setup**                        


In [ ]:
# Install qwen-tts package
!pip install -U qwen-tts soundfile -q

# Install flash-attention for faster inference (optional but recommended)
# This compilation can take 5-10 minutes
!pip install flash-attn --no-build-isolation -q

# Verify GPU and imports
import torch
import soundfile as sf
import os
from IPython.display import Audio, display, Markdown, HTML

print(f"PyTorch: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")

# Create output directory
os.makedirs("audio_outputs", exist_ok=True)

# Check flash attention availability
try:
    import flash_attn
    ATTN_IMPL = "flash_attention_2"
    print("\n✅ Flash Attention 2 available")
except ImportError:
    ATTN_IMPL = "eager"
    print("\n⚠️ Flash Attention not available, using standard attention")

    # Helper function for generating and playing audio
def play_audio(wav, sr, filename=None, title=None):
    """Save and display audio with optional title"""
    if filename:
        filepath = f"audio_outputs/{filename}.wav"
        sf.write(filepath, wav, sr)
    if title:
        display(Markdown(f"**{title}**"))
    display(Audio(wav, rate=sr))

**2. Model Training**

In [ ]:
from qwen_tts import Qwen3TTSModel
clone_model = Qwen3TTSModel.from_pretrained(
    "Qwen/Qwen3-TTS-12Hz-1.7B-Base",
    device_map="cuda:0",
    dtype=torch.bfloat16,
    attn_implementation=ATTN_IMPL,
)
print("✅ Base model loaded!")

**3. Upload Reference_audio_clip**

In [ ]:
# Voice cloning from URL reference audio
print("🔊 VOICE CLONING DEMO\n")

# Using official sample audio
ref_audio_url = "/content/clove_reference.mp3"
ref_text = "Yeah, what else can I do?"

print(f"📎 Reference audio: {ref_audio_url}")
print(f"📝 Reference transcript: \"{ref_text[:50]}...\"\n")

**4. Fetch generated/clonned voice sample**

In [ ]:
# Generate new content with cloned voice
new_texts = [
    "This is the sample voice audio of clove from colab notebook (GOOGLE)"
]

for i, text in enumerate(new_texts):
    print(f"\n🎙️ Generating: \"{text[:50]}...\"")

    wavs, sr = clone_model.generate_voice_clone(
        text=text,
        language="English",
        ref_audio=ref_audio_url,
        ref_text=ref_text,
    )
    play_audio(wavs[0], sr, f"clone_01_sample_{i+1}")

**5. API integration from the above Voice_model {AI GENERATED}**

In [ ]:
from fastapi import FastAPI, HTTPException
from fastapi.responses import FileResponse
from pydantic import BaseModel
import uvicorn
import soundfile as sf
import os
import torch
from qwen_tts import Qwen3TTSModel

app = FastAPI()

# 1. LOAD THE MODEL LOCALLY INTO YOUR GPU
MODEL_PATH = "./Qwen3-TTS-12Hz-1.7B-CustomVoice"
print("[System] Loading Qwen3-TTS into your local NVIDIA GPU...")

clone_model = Qwen3TTSModel.from_pretrained(
    MODEL_PATH,
    device_map="cuda:0",
    torch_dtype=torch.bfloat16,
    attn_implementation="sdpa" # Native PyTorch attention maps
)

class TTSRequest(BaseModel):
    text: str

@app.post("/synthesize")
def synthesize_audio(request: TTSRequest):
    try:
        output_filename = "local_output.wav"
        
        # Hardcode your reference paths on your local PC drive
        ref_url = "C:/Users/sharm/Desktop/CloveAI/ref_audio.mp3" 
        ref_txt = "Well it all started when I was born and then uh. You know what? I'll just tell you later."

        # Generate audio using your local GPU
        wavs, sr = clone_model.generate_voice_clone(
            text=request.text,
            language="English",
            ref_audio=ref_url,
            ref_text=ref_txt
        )
        
        audio_data = wavs[0] if isinstance(wavs, (list, tuple)) else wavs
        if hasattr(audio_data, 'cpu'):
            audio_data = audio_data.cpu().numpy()
            
        sf.write(output_filename, audio_data, sr)
        return FileResponse(output_filename, media_type="audio/wav")
    except Exception as e:
        raise HTTPException(status_code=500, detail=str(e))

if __name__ == "__main__":
    # Run locally on localhost port 8000 (No Ngrok needed!)
    uvicorn.run(app, host="127.0.0.1", port=8000)